# Medical Cost Personal Prediction

**Tabular Regression Project** — Predict medical insurance costs.

Models: CatBoost · LightGBM · XGBoost · FLAML · LazyPredict
Dataset: Medical Cost Personal (local `insurance.csv`, 1,338 rows)
Target: `charges`

## 2 · Project Overview

We predict **medical insurance charges** using the classic insurance dataset. With 1,338 rows and 6 features, this clean dataset demonstrates how a single dominant feature (smoking status) can drive regression predictions.

## 3 · Learning Objectives

1. Work with a clean, small medical cost dataset.
2. Discover feature dominance patterns.
3. Create interaction features for non-linear relationships.
4. Handle skewed targets.
5. Compare multiple regression approaches.

## 4 · Problem Statement

Predict **annual medical insurance charges** from age, sex, BMI, children, smoking status, and region.

## 5 · Why This Project Matters

- Insurance pricing directly impacts millions of people.
- Understanding cost drivers helps with healthcare planning.
- Demonstrates clear non-linear feature interactions.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Source** | Local: `insurance.csv` |
| **Rows** | 1,338 |
| **Features** | age, sex, bmi, children, smoker, region |
| **Target** | charges (USD) |

## 7 · Dataset Source and License

- **Source**: Medical Cost Personal Datasets.
- **License**: Public domain.
- **Note**: From a statistics textbook.

## 8 · Environment Setup

In [ ]:
import subprocess, sys
def _install_if_missing(pkg, imp=None):
    try: __import__(imp or pkg)
    except ImportError: subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
for p, i in [('catboost',None),('lightgbm',None),('xgboost',None),('flaml',None),('lazypredict',None)]:
    _install_if_missing(p, i)
print('All packages ready.')

## 9 · Imports

In [ ]:
import os, warnings, json
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
try:
    from sklearn.metrics import root_mean_squared_error
except ImportError:
    from sklearn.metrics import mean_squared_error
    def root_mean_squared_error(y_true, y_pred): return mean_squared_error(y_true, y_pred, squared=False)
warnings.filterwarnings('ignore')
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print('Imports complete.')

## 10 · Configuration / Constants

In [ ]:
SEED = 42
TEST_SIZE = 0.2
TARGET = 'charges'
np.random.seed(SEED)

## 11 · Dataset Loading

In [ ]:
data_file = os.path.join(SAVE_DIR, 'insurance.csv')
assert os.path.exists(data_file), f'Not found: {data_file}'
df = pd.read_csv(data_file)
print(f'Loaded: {df.shape}')
df.head()

## 12 · Data Validation

In [ ]:
print('Missing:', df.isnull().sum().sum())
print(f'Duplicates: {df.duplicated().sum()}')
print(f'Target mean: ${df[TARGET].mean():,.0f}')

## 13 · Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
df[TARGET].hist(bins=50, ax=axes[0,0], edgecolor='black')
axes[0,0].set_title('Charges Distribution')
axes[0,1].scatter(df['age'], df[TARGET], c=df['smoker'].map({'yes':1,'no':0}), alpha=0.4, cmap='coolwarm')
axes[0,1].set_xlabel('Age'); axes[0,1].set_ylabel('Charges')
axes[0,1].set_title('Age vs Charges (red=smoker)')
df.boxplot(column=TARGET, by='smoker', ax=axes[1,0])
axes[1,0].set_title('Charges by Smoker'); plt.suptitle('')
axes[1,1].scatter(df['bmi'], df[TARGET], alpha=0.3)
axes[1,1].set_xlabel('BMI'); axes[1,1].set_ylabel('Charges')
axes[1,1].set_title('BMI vs Charges')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_plots.png'), dpi=100)
plt.show()

## 14 · Target Analysis

In [ ]:
print(df[TARGET].describe())
print(f'\nSkewness: {df[TARGET].skew():.2f}')

## 15 · Train / Validation / Test Split

## 16 · Preprocessing

In [ ]:
from sklearn.preprocessing import LabelEncoder
for col in ['sex', 'smoker', 'region']:
    df[col] = LabelEncoder().fit_transform(df[col])
print(f'Preprocessed: {df.shape}')

## 17 · Feature Engineering

In [ ]:
df['smoker_bmi'] = df['smoker'] * df['bmi']
df['smoker_age'] = df['smoker'] * df['age']
df['age_bmi'] = df['age'] * df['bmi']
X = df.drop(columns=[TARGET])
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## 18 · Baseline Model

In [ ]:
baseline = LinearRegression()
baseline.fit(X_train, y_train)
y_pred_base = baseline.predict(X_test)
baseline_rmse = root_mean_squared_error(y_test, y_pred_base)
baseline_mae = mean_absolute_error(y_test, y_pred_base)
baseline_r2 = r2_score(y_test, y_pred_base)
print(f'Baseline LR: RMSE={baseline_rmse:,.0f}, MAE={baseline_mae:,.0f}, R2={baseline_r2:.4f}')

## 19 · LazyPredict Benchmark

In [ ]:
from lazypredict.Supervised import LazyRegressor
lazy = LazyRegressor(verbose=0, ignore_warnings=True, random_state=SEED)
lazy_models, _ = lazy.fit(X_train.head(min(5000,len(X_train))), X_test.head(min(1000,len(X_test))),
                           y_train.head(min(5000,len(y_train))), y_test.head(min(1000,len(y_test))))
print(lazy_models.head(15))

## 20 · FLAML AutoML Run

In [ ]:
try:
    from flaml import AutoML
    flaml_model = AutoML()
    flaml_model.fit(X_train, y_train, task='regression', time_budget=60, metric='rmse', seed=SEED, verbose=0)
    y_pred_flaml = flaml_model.predict(X_test)
    flaml_rmse = root_mean_squared_error(y_test, y_pred_flaml)
    flaml_mae = mean_absolute_error(y_test, y_pred_flaml)
    flaml_r2 = r2_score(y_test, y_pred_flaml)
    print(f'FLAML best: {flaml_model.best_estimator}')
    print(f'  RMSE: {flaml_rmse:.2f}, MAE: {flaml_mae:.2f}, R2: {flaml_r2:.4f}')
except Exception as e:
    print(f'FLAML failed: {e}')
    flaml_rmse = flaml_mae = flaml_r2 = None

## 21 · Boosting Models

In [ ]:
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
models = {
    'CatBoost': CatBoostRegressor(iterations=300, learning_rate=0.1, depth=6, random_seed=SEED, verbose=0),
    'LightGBM': LGBMRegressor(n_estimators=300, learning_rate=0.1, max_depth=6, random_state=SEED, verbose=-1),
    'XGBoost': XGBRegressor(n_estimators=300, learning_rate=0.1, max_depth=6, random_state=SEED, verbosity=0)
}
results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    results[name] = {
        'RMSE': root_mean_squared_error(y_test, preds),
        'MAE': mean_absolute_error(y_test, preds),
        'R2': r2_score(y_test, preds)
    }
    print(f'{name}: RMSE={results[name]["RMSE"]:.2f}, MAE={results[name]["MAE"]:.2f}, R2={results[name]["R2"]:.4f}')

## 22 · Top 3 Model Selection

In [ ]:
all_results = {}
all_results['Baseline_LR'] = {'RMSE': baseline_rmse, 'MAE': baseline_mae, 'R2': baseline_r2}
if flaml_r2 is not None:
    all_results['FLAML'] = {'RMSE': flaml_rmse, 'MAE': flaml_mae, 'R2': flaml_r2}
all_results.update(results)
ranked = sorted(all_results.items(), key=lambda x: x[1]['RMSE'])
print('All models ranked by RMSE:')
for name, m in ranked:
    print(f"  {name:20s}  RMSE={m['RMSE']:.2f}  MAE={m['MAE']:.2f}  R2={m['R2']:.4f}")
top3_names = [r[0] for r in ranked[:3]]
print(f'\nTop 3: {top3_names}')

## 23 · Final Eval of Top 3

In [ ]:
print('Top 3 Final Results:')
print('=' * 60)
for name in top3_names:
    m = all_results[name]
    print(f"{name}: RMSE={m['RMSE']:.2f}, MAE={m['MAE']:.2f}, R2={m['R2']:.4f}")

## 24 · Error Analysis

In [ ]:
best_name = top3_names[0]
if best_name in models: best_model = models[best_name]
else: best_model = models.get('CatBoost', baseline)
y_pred_best = best_model.predict(X_test)
residuals = y_test.values - y_pred_best
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].scatter(y_pred_best, residuals, alpha=0.5); axes[0].axhline(0, color='r', linestyle='--')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Residual'); axes[0].set_title('Residuals vs Predicted')
axes[1].hist(residuals, bins=30, edgecolor='black'); axes[1].set_title('Residual Distribution')
axes[2].scatter(y_test, y_pred_best, alpha=0.5)
axes[2].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
axes[2].set_xlabel('Actual'); axes[2].set_ylabel('Predicted'); axes[2].set_title('Actual vs Predicted')
plt.tight_layout(); plt.savefig(os.path.join(SAVE_DIR, 'error_analysis.png'), dpi=100); plt.show()

## 25 · Interpretation

- **Smoker status** overwhelmingly dominates (3-4x higher charges).
- **smoker_bmi** interaction captures amplified risk.
- **Age** is the secondary driver.
- Bimodal distribution reflects smoker/non-smoker split.

## 26 · Limitations

- Only 1,338 rows.
- Binary smoker status oversimplifies.
- No pre-existing conditions or medical history.
- Synthetic/textbook data.

## 27 · How to Improve

1. Log-transform charges.
2. Separate models for smokers/non-smokers.
3. Add health history.
4. BMI categories.

## 28 · Production Considerations

- Regulatory compliance for insurance pricing.
- Anti-discrimination laws.
- Model explainability requirements.
- Actuarial validation.

## 29 · Common Mistakes

1. Not creating smoker x BMI interaction.
2. Ignoring bimodal distribution.
3. Using only linear models.
4. Not recognizing smoker dominance.

## 30 · Mini Challenge

1. Predict log(charges).
2. Build separate smoker/non-smoker models.
3. Add BMI categories.
4. SHAP explanation.

## 31 · Final Summary

- Smoker status dominates insurance cost prediction.
- Interaction features critical for linear models.
- Boosting models learn interactions automatically.
- Classic dataset for regression fundamentals.

In [ ]:
metrics = {}
for name in top3_names: metrics[name] = all_results[name]
metrics['baseline'] = all_results['Baseline_LR']
with open(os.path.join(SAVE_DIR, 'metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)
print('Metrics saved to metrics.json')
print('\nNotebook complete.')